# MDTF Variable Translation: Developer Guide

This notebook introduces the framework's variable-name translation machinery. It is for code exploration and small examples; it does not launch a full MDTF run or modify the fieldlist files. Run it from a kernel with the MDTF repository dependencies available.

For a reusable template that you can adapt for future POD examples, see [variable_translation_template.ipynb](variable_translation_template.ipynb).

> Disclaimer: This notebook was created with GitHub Copilot assistance and has been vetted for accuracy.

## The Main Idea

POD settings describe requested variables. Fieldlist tables describe names and metadata used by data conventions. The translator loads those tables and routes lookups to the fieldlist for a convention. In a framework run, `translate_data` decides whether the data convention is used or whether the special `no_translation` fieldlist is selected.

The main runtime path is:

1. [`mdtf_framework.py`](../../mdtf_framework.py#L159) loads translator tables and [`mdtf_framework.py`](../../mdtf_framework.py#L185) starts POD setup.
2. [`PodObject.setup_pod`](../../src/pod_setup.py#L313) checks `runtime_config.translate_data` and chooses the target convention.
3. [`DataSourceBase.translate_varlist`](../../src/data_sources.py#L100) forwards the variable to its varlist.
4. [`Varlist.setup_var`](../../src/varlist_util.py#L547) selects a fieldlist and calls that fieldlist's `translate()` method.

The `VariableTranslator.translate()` wrapper is a separate convenience method. The gated framework path above calls the selected fieldlist directly. See [`translation.py`](../../src/translation.py#L634) for the wrapper implementation.

## Find The Repository Root

The source package is named `src`. Find the checkout from the notebook's current directory so imports work whether Jupyter started in the repository root or a subdirectory.

This notebook imports MDTF internals directly from the repository's `src` package. Until a lightweight, pip-installable MDTF API is exposed, it must locate the checkout and add its root to `sys.path` so those imports work when Jupyter is started from a subdirectory. This is a temporary development workaround; consumers of the future installed API should not need to manage repository paths.

In [ ]:
import sys
from pathlib import Path

# Find the repository root directory for the MDTF checkout
candidate_dirs = (Path.cwd(), *Path.cwd().parents)
repo_root = next(
    (path for path in candidate_dirs if (path / "src" / "translation.py").is_file()),
    None,
)
if repo_root is None:
    raise RuntimeError("Start Jupyter in the MDTF checkout or one of its subdirectories.")

# Adds the repository root to the system path so that the translation module can be imported
sys.path.insert(0, str(repo_root))
print(repo_root)

## Load the Fieldlists

`VariableTranslator` is a singleton in this codebase. The framework initializes it once with `repo_root`, then reads `data/fieldlist_*.jsonc`. The field lists contain variable attributes for each convention (e.g., CMIP, GFDL, CESM). For an isolated notebook kernel, use the same initialization sequence below.

In [ ]:
from src.translation import VariableTranslator

translator = VariableTranslator(str(repo_root))
translator.read_conventions(str(repo_root))

print("Loaded conventions:", sorted(translator.conventions.keys()))

## Air Temperature Across Conventions

We'll take a look at the variable for air temperature across GFDL, CMIP, and CESM conventions using the variable translator. Certain attributes like the standard name, modifier, and realm all match. Other attributes differ. For example, only GFDL has a long name.

`lut` is short for lookup table. In this code, it’s a dictionary on the convention’s fieldlist: variable names are keys, and their metadata are the values.

In [ ]:
gfdl_t_ref = translator.get_convention("GFDL").lut["t_ref"]
cmip_tas = translator.get_convention("CMIP").lut["tas"]
cesm_TREFHT = translator.get_convention("CESM").lut["TREFHT"]

print(f"CONVENTION VAR:\tstandard_name\tmodifier\trealm\tlong_name")
print("")
print(f"GFDL t_ref:\t{gfdl_t_ref["standard_name"]}\t{gfdl_t_ref["modifier"]}\t{gfdl_t_ref["realm"]}\t{gfdl_t_ref["long_name"]}")
print(f"CMIP tas:\t{cmip_tas["standard_name"]}\t{cmip_tas["modifier"]}\t{cmip_tas["realm"]}\t{cmip_tas["long_name"]}")
print(f"CESM TREFHT:\t{cesm_TREFHT["standard_name"]}\t{cesm_TREFHT["modifier"]}\t{cesm_TREFHT["realm"]}\t{cesm_TREFHT["long_name"]}")

## Translate GFDL POD Variable to Other Conventions

This example starts from a CMIP `VarlistEntry`. The translated versions follow the GFDL and CESM conventions. The matching entries share `air_temperature`, realm `atmos`, units `K`, three dimensions, and modifier `atmos_height`. The translated name should be `t_ref` for GFDL. We'll even throw in `TREFHT` for CESM as part of this example.

For standalone use, `VariableTranslator.translate(var, to_convention, from_convention)` is a wrapper that selects the fieldlist and calls the same method.

In [ ]:
from types import SimpleNamespace
from src.varlist_util import Varlist

# The example_settings are akin to the settings.json in the v4 framework.
# The "t_ref" variable represents the near-surface air temperature in the GFDL convention.
# We're going to try using the variable translation functionality to convert CMIP to GFDL.
example_settings = {
    "data": {"frequency": "day", "realm": "atmos"},
    "dimensions": {
        "lat": {
            "axis": "Y",
            "standard_name": "latitude",
            "units": "degrees_north",
        },
        "lon": {
            "axis": "X",
            "standard_name": "longitude",
            "units": "degrees_east",
        },
        "time": {"axis": "T", "standard_name": "time", "units": "days"},
    },
    "varlist": {
        "t_ref": {
            "standard_name": "air_temperature",
            "long_name": "temperature at 2 m",
            "units": "K",
            "modifier": "atmos_height",
            "dimensions": ["time", "lat", "lon"],
        }
    },
}

# Create a simple namespace object to represent the example pod.
pod = SimpleNamespace(
    name="simple_example_pod",
    pod_data=example_settings["data"],
    pod_dims=example_settings["dimensions"],
    pod_vars=example_settings["varlist"],
    pod_settings={"convention": "GFDL"},
    _log_name="translation_notebook",
)

# Start from the GFDL t_ref variable
gfdl_var = Varlist.from_struct(pod).vars[0]

# Translate from GFDL variable to the CMIP convention
cmip_var = translator.translate(gfdl_var, to_convention="CMIP", from_convention="GFDL")

# Translate from GFDL variable to the CESM convention
cesm_var = translator.translate(gfdl_var, to_convention="CESM", from_convention="GFDL")

print(f"CONV\tname\tstandard_name\tmodifier\trealm\tunits\tlong_name\n")

for var in [gfdl_var, cmip_var, cesm_var]:
    print(f"{var.convention}\t{var.name}\t{var.standard_name}\t{var.modifier}\t{var.realm}\t{var.units}\t{var.long_name}")

## How the Runtime Gate Works

In [`src/pod_setup.py`](../../src/pod_setup.py#L313), each case starts with its configured data convention. If `translate_data` is false, setup substitutes `no_translation`; otherwise the configured data convention remains the target. Setup passes the POD convention and chosen target convention into `translate_varlist()`.

For the example translation here, those values are `pod_convention="GFDL"`, `data_convention="CMIP"`, and `translate_data=True`. The framework therefore selects the CMIP fieldlist and passes the GFDL convention as the POD source.

In [`src/varlist_util.py`](../../src/varlist_util.py#L547), `VariableTranslator().get_convention(to_convention)` returns the data fieldlist, or a `NoTranslationFieldlist` for the sentinel. The code then calls `translate.translate(v, from_convention)`. So the switch affects which fieldlist implementation runs; it does not skip the varlist setup call.

Keep the two APIs distinct when debugging:

- `VariableTranslator.translate(var, to_convention, from_convention)` is a wrapper that chooses a fieldlist and calls its `translate()`.
- The `translate_data` runtime path calls `Fieldlist.translate(var, from_convention)` on the fieldlist already selected in `Varlist.setup_var()`.
- Preprocessor request-editing has a direct `VariableTranslator.translate()` call at [`src/preprocessor.py`](../../src/preprocessor.py#L206); that is another use case.

## Example Using the MDTF Framework Preprocessor Functions to Perform Variable Translation and Unit Conversion

This example reads CMIP-convention `tas` from the catalog named in `case_info_demo2.yml`, translates it to the GFDL `t_ref` variable for each configured case, and applies the framework's unit and coordinate preprocessing. It then follows the multirun demo's calculation and plot pattern: time mean, shuffled-field global-mean anomaly, zonal mean, and a latitude profile for each case.

This variable pair demonstrates the overall preprocessing path, but not a nontrivial unit or coordinate conversion: both conventions specify `K`, and their coordinate names are `time`, `lat`, and `lon`. Thus `ConvertUnitsFunction` makes no numerical change and `RenameVariablesFunction` has no coordinate labels to change. The dependent-variable rename from `tas` to `t_ref` is separate because the existing `RenameVariablesFunction` only renames coordinates. The shuffle is only a visual demonstration and should not be interpreted scientifically. Catalog assets are opened lazily and only the reduced latitude profiles are loaded into memory.

In [ ]:
from types import SimpleNamespace

from src.preprocessor import ConvertUnitsFunction, RenameVariablesFunction
from src.varlist_util import Varlist

# The POD convention is GFDL and the data catalog convention is CMIP.
POD_CONVENTION = "GFDL"
DATA_CATALOG_CONVENTION = "CMIP"

# We're looking at the GFDL t_ref variable for this example and using the translator to access its 
# attributes, like standard_name, long_name, etc. for convenience purposes, rather than explicitly defining them.
POD_VAR_NAME = "t_ref"
pod_field = translator.get_convention(POD_CONVENTION).to_CF(POD_VAR_NAME)
runtime_settings = {
    "data": {"frequency": "day", "realm": pod_field.get("realm") or "atmos"},
    "dimensions": {
        "lat": {
            "axis": "Y",
            "standard_name": "latitude",
            "units": "degrees_north",
        },
        "lon": {
            "axis": "X",
            "standard_name": "longitude",
            "units": "degrees_east",
        },
        "time": {"axis": "T", "standard_name": "time", "units": "days"},
    },
    "varlist": {
        POD_VAR_NAME: {
            "standard_name": pod_field["standard_name"],
            "long_name": pod_field.get("long_name", ""),
            "units": pod_field["units"],
            "modifier": pod_field.get("modifier", ""),
            "dimensions": ["time", "lat", "lon"],
        }
    },
}
runtime_pod = SimpleNamespace(
    name="framework_preprocessing_example",
    pod_data=runtime_settings["data"],
    pod_dims=runtime_settings["dimensions"],
    pod_vars=runtime_settings["varlist"],
    pod_settings={"convention": POD_CONVENTION},
    _log_name="framework_preprocessing_example",
)
# This example uses a single variable, but in practice, there may be multiple variables to translate.
pod_var = Varlist.from_struct(runtime_pod).vars[0] # Reminder: GFDL t_ref

# Get the translation to CMIP
pod_var.translation = translator.translate(pod_var, to_convention=DATA_CATALOG_CONVENTION, from_convention=POD_CONVENTION)
translated_name = pod_var.translation.name

# Alternative way to translate to CMIP
#pod_var.translation = translator.get_convention(DATA_CATALOG_CONVENTION).translate(pod_var, POD_CONVENTION)

print(f"{POD_CONVENTION} POD variable: {pod_var.name}; {DATA_CATALOG_CONVENTION} data variable: {translated_name}")

### Load CMIP tas data through the case-info catalog

Read the catalog referenced by `case_info_demo2.yml`. The data is in CMIP convention.

Determine the CMIP convention equivalent for GFDL `t_ref`. Then query for that CMIP variable information from the data catalog. Convert the queried CMIP data for each configured case to the GFDL equivalent.

In [ ]:
from pathlib import Path
import intake
import yaml

example_dir = repo_root / "diagnostics" / "example_multicase"
case_info_path = example_dir / "case_info_demo2.yml"
with case_info_path.open() as stream:
    case_info = yaml.safe_load(stream)

configured_cases = case_info["CASE_LIST"]
first_case = next(iter(configured_cases.values()))
time_coord = first_case["time_coord"]
lon_coord = first_case["lon_coord"]

# Use the catalog named by the case-info file, with its colocated copy as a portable fallback.
catalog_path = Path(case_info["CATALOG_FILE"]).expanduser()
if not catalog_path.is_file():
    catalog_path = example_dir / catalog_path.name
if not catalog_path.is_file():
    raise FileNotFoundError(f"Case-info catalog not found: {catalog_path}")

cat = intake.open_esm_datastore(str(catalog_path))
var_cat = cat.search(variable_id=translated_name, frequency="daily")
if var_cat.df.empty:
    raise ValueError(f"No daily catalog entries found for {DATA_CATALOG_CONVENTION}")

var_cat

In [ ]:
var_cat.df

### Run framework preprocessing

Convert units and reconcile translated coordinate names with `ConvertUnitsFunction` and `RenameVariablesFunction`. Both of these classes perform conversions through their `execute` functions and return transformed `xarray.Dataset` objects.

In [ ]:
catalog_datasets = {}
gfdl_datasets = {}
# Iterate over each configured case and translate the CMIP data to GFDL format.
for case_key, case_config in configured_cases.items():
    configured_case_name = case_config["case_name"]
    experiment_ids = [
        experiment_id
        for experiment_id in var_cat.df["experiment_id"].dropna().unique()
        if str(experiment_id) in configured_case_name
    ]
    if len(experiment_ids) != 1:
        raise ValueError(
            f"Expected one catalog experiment for {configured_case_name!r}; "
            f"found {experiment_ids}"
        )

    case_var_catalog = cat.search(
        variable_id=translated_name,
        frequency="daily",
        experiment_id=experiment_ids[0],
    )
    case_datasets = case_var_catalog.to_dataset_dict(
        xarray_open_kwargs={"decode_times": True, "use_cftime": True}
    )
    
    if len(case_datasets) != 1:
        raise ValueError(
            f"Expected one daily CMIP dataset for {configured_case_name!r}; "
            f"found {len(case_datasets)}"
        )
    cmip_dataset, = case_datasets.values() # Take the first value. Only one is expected.
    catalog_datasets[case_key] = cmip_dataset

    # Execute the Preprocessor functions
    gfdl_dataset = ConvertUnitsFunction().execute(pod_var, cmip_dataset.copy(deep=False))
    gfdl_dataset = RenameVariablesFunction().execute(pod_var, gfdl_dataset)

    # Rename the variable if the translated name differs from the POD variable name.
    # The RenameVariablesFunction has the following step commented out, but we need it here
    # to ensure the variable is correctly named in the gfdl_dataset.
    if translated_name != pod_var.name:
        gfdl_dataset = gfdl_dataset.rename_vars({translated_name: pod_var.name})

    gfdl_datasets[case_key] = gfdl_dataset
    print(
        f"{configured_case_name}: CMIP {translated_name} -> GFDL {pod_var.name}; "
        f"sizes={dict(gfdl_dataset.sizes)}"
    )

### Check that the POD variable is now in the converted datasets

In [ ]:
pod_named_datasets = gfdl_datasets
for case_name, dataset in pod_named_datasets.items():
    assert pod_var.name in dataset.data_vars
    print (dataset.data_vars)
    print(f"{case_name}: CMIP {translated_name} -> GFDL {pod_var.name}; sizes={dict(dataset.sizes)}")

### Calculate Shuffled Zonal-mean Anomalies

Like the multirun demo, randomly shuffle each case's time-mean field before calculating anomalies and zonal means. This makes the example plot visually interesting, but destroys the geographic pattern; do not interpret these shuffled profiles as scientific results. This is for example purposes only.

In [ ]:
import numpy as np

np.random.seed(0)
t_ref_arrays = {}
for catalog_key, dataset in pod_named_datasets.items():
    t_ref = dataset[pod_var.name].mean(dim=time_coord)

    # Match the multirun demo: shuffle the time-mean field before anomalies.
    t_ref.load()
    shuffled_values = t_ref.to_masked_array().flatten()
    np.random.shuffle(shuffled_values)
    t_ref.values = shuffled_values.reshape(t_ref.shape)

    t_ref_anomaly = t_ref - t_ref.mean()
    t_ref_zonal_mean = t_ref_anomaly.mean(dim=lon_coord)
    t_ref_arrays[catalog_key] = t_ref_zonal_mean
    print(f"Calculated shuffled zonal-mean anomaly for {catalog_key}")

for dataset in catalog_datasets.values():
    dataset.close()

cat.close()
t_ref_arrays

### Plot zonal-mean anomalies

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display

fig, ax = plt.subplots(figsize=(12, 4))
for case_name, latitude_profile in t_ref_arrays.items():
    latitude_profile.plot(ax=ax, label=case_name)

ax.set_title("Zonal Mean Surface Air Temperature Anomaly")
ax.set_xlabel("Latitude")
ax.set_ylabel(f"{pod_var.name} anomaly ({pod_var.units})")
ax.legend(title="GFDL case")
plt.tight_layout()